### Identify encoder failures through time

Do EFD study of locations and frequency of each encoder (az and el) to determine problematic spots, and degradation rates before we have operational issues like this week. 

There are warning logmessages when encoders drop out, so it should be a simple matter to match those to the azimuth/elevation position then finally tape position.

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from astropy.time import Time
from pathlib import Path
from datetime import datetime

from lsst.summit.utils.tmaUtils import (
    TMAEventMaker,
    TMAState,
)
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime

## Create Event Maker


In [ ]:
event_maker = TMAEventMaker()
client = makeEfdClient()

In [ ]:
# make a list of all topics in the EFD related to MTMount
topics = await client.get_topics()
for topic in topics:
    if 'MTMount' in topic:
        print(topic)

In [ ]:
# Get all azimuth position and timestamp data within a particular time range
start = Time("2025-12-01T00:00:00Z", scale="utc")
end = Time("2025-12-01T23:59:00Z", scale="utc")

In [ ]:
error = await client.select_time_series(
    "lsst.sal.MTMount.logevent_error", ["*"], start, end
)

In [ ]:
error

In [ ]:
counts = error['text'].value_counts()

print(counts)

plt.figure(figsize=(10, 6))
counts.plot(kind='bar')

plt.xlabel("Error type")
plt.ylabel("Count")
plt.title("Error frequency")
plt.grid(axis="y", linestyle=":", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
specific_error = """VI chain: Interpreter.lvlib:SplitString.vi:3850525 in RemoteVariableServer.lvlib:Task.vi.ACBRProxyCaller.4C500020->RemoteVariableServer.lvlib:Task.vi:5130002->RemoteVariableServer.lvlib:CreateVariableSubscriptionBinary.vi:5800003->RemoteVariableServer.lvlib:GetReadTaskVariablesInfoForBinary.vi:5120007<PARSED_NL> Description: LabVIEW: (Hex 0x2A) Generic error.<PARSED_NL>=========================<PARSED_NL>Custom:  Invalid equation!<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty expression: "TMA_AZ_ENC_ELV_0001_ErrorStatus"<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty Event: "Name: Elevation Encoder Head 1 lost. Area 700.EIB. Event type: "<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty Event: "Name: Elevation Encoder Head 2 lost. Area 700.EIB. Event type: "<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty Event: "Name: Elevation Encoder Head 3 lost. Area 700.EIB. Event type: "<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty Event: "Name: Elevation Encoder Head 4 lost. Area 700.EIB. Event type: "<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty Event: "Name: Azimuth Encoder Head 1 lost. Area 700.EIB. Event type: "<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty Event: "Name: Azimuth Encoder Head 2 lost. Area 700.EIB. Event type: "<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty Event: "Name: Azimuth Encoder Head 3 lost. Area 700.EIB. Event type: "<PARSED_NL>=========================<PARSED_NL>Custom:  Faulty Event: "Name: Azimuth Encoder Head 4 lost. Area 700.EIB. Event type: "<PARSED_NL>=========================<PARSED_NL>Origin:  TMA controller Event Handling Loop"""

In [ ]:
tabla_error_VI_chain = error[ error['text'] == specific_error ]
print(tabla_error_VI_chain)

In [ ]:
az = await client.select_time_series(
    "lsst.sal.MTMount.azimuth", ["actualPosition", "timestamp"], start, end
)

In [ ]:
az.index = pd.to_datetime(az.index)
tabla_error_VI_chain.index = pd.to_datetime(tabla_error_VI_chain.index)

In [ ]:
az['timestamp'] = pd.to_datetime(az['timestamp'], unit='s')
az.set_index('timestamp', inplace=True)

In [ ]:
error_times = tabla_error_VI_chain.index

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(az.index, az['actualPosition'], label="Az position")

# Puntos de error (en rojo)
plt.scatter(
    error_times,
    az.loc[error_times, 'actualPosition'],   # la posición en esos instantes
    color='red',
    s=50,
    label='Errors'
)

plt.xlabel("Time")
plt.ylabel("Az position")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
az

In [ ]:
el = await client.select_time_series(
    "lsst.sal.MTMount.elevation", ["actualPosition", "timestamp"], start, end
)

In [ ]:
times_az = [datetime.datetime.utcfromtimestamp(ts) for ts in az['timestamp']]
times_el = [datetime.datetime.utcfromtimestamp(ts) for ts in el['timestamp']]

fig, ax = plt.subplots()

ax.plot(
    times_az,
    az['actualPosition'],
    color="blue",
    label="Az",
)

ax.plot(
    times_el,
    el['actualPosition'],
    color="orange",
    label="El",
)

ax.legend(loc="lower left")
ax.grid(":", alpha=0.25)

# EJE X: Time
ax.set_xlabel("Time")

# EJE Y: vacío (sin texto)
ax.set_ylabel("")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M:%S"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

plt.gcf().autofmt_xdate()  # gira las fechas para que no se solapen

plt.tight_layout()
plt.show()

In [ ]:
encoder = await client.select_time_series(
    "lsst.sal.MTMount.encoder", ["*"], start, end
)

In [ ]:
encoder.columns